<a href="https://colab.research.google.com/github/juanpajaro/aprendizaje_profundo_salud_puj_2026/blob/main/Taller_1_solucion_DL_en_salud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import keras
import pandas as pd
import numpy as np
import re
from keras.preprocessing.sequence import pad_sequences
from keras import layers
import matplotlib.pyplot as plt

In [ ]:
datos = pd.read_csv('/content/datos_apnea.csv')
datos.head()

In [ ]:
datos["Apnea"].value_counts()

In [ ]:
#count NaN in datos["Apnea"]
datos["Apnea"].isna().sum()

In [ ]:
#convert NaN in datos["Apnea"] to 1 and "No" to cero
datos["Apnea"] = datos["Apnea"].fillna(1)
datos["Apnea"] = datos["Apnea"].replace("No", 0)
datos["Apnea"].value_counts()

In [ ]:
text_raw = datos["EnfermedadActual"].to_numpy()
print(type(text_raw))
y_raw = datos["Apnea"].to_numpy()
print(type(y_raw))

In [ ]:
print(len(text_raw))
print(text_raw.shape)

In [ ]:
print(text_raw[1])

In [ ]:
def encontrar_palabras(oracion):
  #reemplaza signos de puntuacion por puntos
  oracion_limpia = re.sub(r'[,!?;-]', '.', oracion)
  #convertir la oración en minuscula
  oracion_minuscula = oracion_limpia.lower()
  #eliminar comillas dobles y simples
  oracion_sin_comilla = re.sub(r'["\']', '', oracion_minuscula)
  #se eliminan espacios
  l_oracion = oracion_sin_comilla.split()
  oracion_final = [palabra.strip() for palabra in l_oracion]
  return " ".join(oracion_final)

In [ ]:
#Ejemplo de ejecución y resultado de la función anterior
for i in text_raw[:2]:
  prueba = encontrar_palabras(i)
  print(prueba)

In [ ]:
for i in range(len(text_raw)):
  text_raw[i] = encontrar_palabras(text_raw[i])

print(text_raw[1])

In [ ]:
lp_ora = text_raw[1].split()
print(lp_ora)
print(list(set(lp_ora)))

In [ ]:
def obtener_diccionario(oraciones, verbose=False):
  palabraAindice = {}
  indiceApalabra = {}
  indice = 0
  for oracion in oraciones:
    l_oracion = oracion.split()
    palabras_unicas_en_oracion = list(set(l_oracion))
    if verbose:
      print(l_oracion)
    for palabra in palabras_unicas_en_oracion:
      if palabra not in palabraAindice:
        palabraAindice[palabra] = indice
        indiceApalabra[indice] = palabra
        indice += 1
  return palabraAindice, indiceApalabra

In [ ]:
#Ejemplo de ejecución y resultado de la función anterior
prueba_1, prueba_2 = obtener_diccionario(text_raw[:2], True)
print(prueba_1)
print(prueba_2)
print(len(prueba_1))
print(len(prueba_2))

In [ ]:
#se ejecuta sobre todo el texto
palabraAindice, indiceApalabra = obtener_diccionario(text_raw, False)
print(len(palabraAindice))
print(len(indiceApalabra))

In [ ]:
def convertir_oraciones_indeces(oracion, palabraAindice, verbose=False):
  l_indices = []
  for palabra in oracion.split():
    if verbose:
      print(palabra)
    l_indices = l_indices + [palabraAindice[palabra]]
  return np.array(l_indices)

In [ ]:
#Ejemplo de ejecución y resultado de la función anterior
for i in text_raw[:2]:
  prueba = convertir_oraciones_indeces(i, palabraAindice, False)
  print(prueba)


In [ ]:
sequences = []
for i in range(len(text_raw)):
  texto_por_oracion = convertir_oraciones_indeces(text_raw[i], palabraAindice, verbose=False)
  sequences.append(texto_por_oracion)

# Rellenar secuencias para asegurar una longitud uniforme
# Puedes elegir una longitud máxima (logitud_oracion_max) o determinarla dinámicamente, por ejemplo, max(len(s) para s en secuencias)
# Para este ejemplo, encontremos la longitud máxima entre todas las secuencias
logitud_oracion_max = max(len(s) for s in sequences) if sequences else 0
print(logitud_oracion_max)
todo_texto_indice_padded = pad_sequences(sequences, maxlen=logitud_oracion_max, padding='post')

# Impresion de la primera oracion en indices
print(todo_texto_indice_padded[1])
print(type(todo_texto_indice_padded))


In [ ]:
print(type(todo_texto_indice_padded))
print(todo_texto_indice_padded.shape)
print(type(y_raw))
print(y_raw.shape)

In [ ]:
def multi_hot_encode(sequences, num_classes):
    results = np.zeros((len(sequences), num_classes))
    for i, sequence in enumerate(sequences):
        # Filter indices to be within the valid range [0, num_classes - 1]
        valid_indices = sequence[sequence < num_classes]
        results[i][valid_indices] = 1.0
    return results

In [ ]:
train_data = todo_texto_indice_padded[:70000]
test_data = todo_texto_indice_padded[30000:]
x_train = multi_hot_encode(train_data, num_classes=5000)
x_test = multi_hot_encode(test_data, num_classes=5000)

In [ ]:
x_train[:3]

In [ ]:
x_train.shape

In [ ]:
train_labels = y_raw[:70000]
test_labels = y_raw[30000:]
y_train = train_labels.astype("float32")
y_test = test_labels.astype("float32")

In [ ]:
model = keras.Sequential(
    [
        layers.Dense(16, activation="relu"),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ]
)

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

In [ ]:
x_val = x_train[:10000]
partial_x_train = x_train[10000:]
y_val = y_train[:10000]
partial_y_train = y_train[10000:]

In [ ]:
history = model.fit(
    partial_x_train,
    partial_y_train,
    epochs=20,
    batch_size=512,
    validation_data=(x_val, y_val),
)

In [ ]:
history_dict = history.history
history_dict.keys()

In [ ]:
history_dict = history.history
loss_values = history_dict["loss"]
val_loss_values = history_dict["val_loss"]
epochs = range(1, len(loss_values) + 1)
plt.plot(epochs, loss_values, "r--", label="Training loss")
plt.plot(epochs, val_loss_values, "b", label="Validation loss")
plt.title("[IMDB] Training and validation loss")
plt.xlabel("Epochs")
plt.xticks(epochs)
plt.ylabel("Loss")
plt.legend()
plt.show()

In [ ]:
plt.clf()
acc = history_dict["accuracy"]
val_acc = history_dict["val_accuracy"]
plt.plot(epochs, acc, "r--", label="Training acc")
plt.plot(epochs, val_acc, "b", label="Validation acc")
plt.title("[IMDB] Training and validation accuracy")
plt.xlabel("Epochs")
plt.xticks(epochs)
plt.ylabel("Accuracy")
plt.legend()
plt.show()

In [ ]:
model = keras.Sequential(
    [
        layers.Dense(16, activation="relu"),
        layers.Dense(16, activation="relu"),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ]
)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

x_val = x_train[:10000]
partial_x_train = x_train[10000:]
y_val = y_train[:10000]
partial_y_train = y_train[10000:]

history = model.fit(
    partial_x_train,
    partial_y_train,
    epochs=20,
    batch_size=512,
    validation_data=(x_val, y_val),
)

In [ ]:
history_dict = history.history
loss_values = history_dict["loss"]
val_loss_values = history_dict["val_loss"]
epochs = range(1, len(loss_values) + 1)
plt.plot(epochs, loss_values, "r--", label="Training loss")
plt.plot(epochs, val_loss_values, "b", label="Validation loss")
plt.title("[IMDB] Training and validation loss")
plt.xlabel("Epochs")
plt.xticks(epochs)
plt.ylabel("Loss")
plt.legend()
plt.show()